<!-- # Trevisan's extractor による乱数抽出 -->

# Randomness extraction with Trevisan's extractor

<!-- ## このノートブックで行うこと -->

## What this notebook does

<!-- 量子乱数生成器（QRNG）の raw data から、**Trevisan's extractor** で一様乱数を取り出します。構成は次の論文の、RSH 1 ビット抽出器とブロック weak design（$r = 1$）です。 -->

We extract uniform random numbers from the raw data of a quantum random number generator (QRNG) with **Trevisan's extractor**. The construction is the RSH one-bit extractor with a block weak design ($r = 1$) from the following paper.

> W. Mauerer, C. Portmann, and V. B. Scholz, "A modular framework for randomness extraction based on Trevisan's construction," arXiv:1212.0520 (2012).

<!--
1. 入力のビット列を用意し、1 ビットあたりの min-entropy $h$ を決める（2.）
2. 有限体の演算・1 ビット抽出器・weak design の関数を定義する（3.〜5.）
3. 出力の長さ $m$ やシードの長さ $d$ などのパラメータを決める（6.）
4. シードを作って抽出する（7.）
5. 抽出結果を保存する（8.）
-->

1. Prepare the input bit string and determine the min-entropy per bit $h$ (2.)
2. Define the functions for finite-field arithmetic, the one-bit extractor, and the weak design (3.–5.)
3. Determine the parameters, such as the output length $m$ and the seed length $d$ (6.)
4. Generate the seed and extract (7.)
5. Save the extraction results (8.)

---

<!-- ## 手法 -->

## Methods

### A. Trevisan's extractor

<!-- $n$ ビットの入力 $x$ と $d$ ビットのシード $y$ から、$m$ ビットを次のように出力します（論文の式 (1)）。 -->

From an $n$-bit input $x$ and a $d$-bit seed $y$, $m$ bits are output as follows (Eq. (1) of the paper).

$$
\mathrm{Ext}(x, y) = C(x, y_{S_1})\, C(x, y_{S_2}) \cdots C(x, y_{S_m})
$$

<!--
- $C$：1 ビット抽出器（B）。入力 $x$ と $t$ ビットのシードから 1 ビットを出力する
- $S_1, \ldots, S_m$：シードの位置 $\{0, \ldots, d-1\}$ の部分集合で、それぞれ $t$ 個の要素をもつ（weak design、C）
- $y_{S_i}$：シード $y$ のうち $S_i$ の位置のビットを並べた $t$ ビット
-->

- $C$: the one-bit extractor (B). It outputs one bit from the input $x$ and a $t$-bit seed
- $S_1, \ldots, S_m$: subsets of the seed positions $\{0, \ldots, d-1\}$, each with $t$ elements (the weak design, C)
- $y_{S_i}$: the $t$ bits of the seed $y$ at the positions in $S_i$

<!-- **合成の定理**（論文の Sec. III A 1、Lemma B.8）：$C$ が量子的な side information に対して安全な $(k_1, \varepsilon_1)$-強い 1 ビット抽出器で、$S_1, \ldots, S_m$ が weak $(m, t, r, d)$-design なら、$\mathrm{Ext}$ は量子的な side information に対して安全な $(k_1 + rm,\ m\varepsilon_1)$-強い抽出器です。$r = 1$ の design を使うと、出力 $m$ ビットに必要な入力の min-entropy は $k_1 + m$ です。 -->

**Composition theorem** (Sec. III A 1 and Lemma B.8 of the paper): if $C$ is a $(k_1, \varepsilon_1)$-strong one-bit extractor secure against quantum side information and $S_1, \ldots, S_m$ is a weak $(m, t, r, d)$-design, then $\mathrm{Ext}$ is a $(k_1 + rm,\ m\varepsilon_1)$-strong extractor secure against quantum side information. With a design with $r = 1$, the min-entropy of the input needed for $m$ output bits is $k_1 + m$.

<!-- ### B. 1 ビット抽出器：RSH（Reed–Solomon 符号と Hadamard 符号の連接） -->

### B. One-bit extractor: RSH (concatenation of a Reed–Solomon code and the Hadamard code)

<!-- **有限体 $\mathrm{GF}(2^\ell)$**：要素は $\ell$ ビットのビット列で、$\mathrm{GF}(2)$ 上の多項式 $a_0 + a_1 X + \cdots + a_{\ell-1}X^{\ell-1}$ とみなします（ビット $j$ が $X^j$ の係数）。足し算はビットごとの XOR、掛け算は多項式の積を $\ell$ 次の既約多項式 $f$ で割った余りです。$f$ は、定数項が 1 の $\ell$ 次多項式を小さい順に調べて最初に見つかった既約多項式とし、既約かどうかは Rabin の判定法 -->

**Finite field $\mathrm{GF}(2^\ell)$**: an element is an $\ell$-bit string, regarded as a polynomial $a_0 + a_1 X + \cdots + a_{\ell-1}X^{\ell-1}$ over $\mathrm{GF}(2)$ (bit $j$ is the coefficient of $X^j$). Addition is bitwise XOR, and multiplication is the product of polynomials reduced modulo an irreducible polynomial $f$ of degree $\ell$. We take $f$ to be the first irreducible polynomial found when checking degree-$\ell$ polynomials with constant term 1 in increasing order, and check irreducibility with Rabin's test,

<!--
$$
X^{2^\ell} \equiv X \pmod{f}, \qquad \gcd\bigl(X^{2^{\ell/p}} - X,\ f\bigr) = 1 \quad (\ell \text{ を割り切るすべての素数 } p)
$$
-->

$$
X^{2^\ell} \equiv X \pmod{f}, \qquad \gcd\bigl(X^{2^{\ell/p}} - X,\ f\bigr) = 1 \quad (\text{for every prime } p \text{ dividing } \ell)
$$

<!-- で調べます。 -->

that is, by checking that both of the conditions above hold.

<!-- **RSH**（論文の Sec. III C 3）： -->

**RSH** (Sec. III C 3 of the paper):

<!--
1. 入力 $x$ を $\ell$ ビットずつ $s = \lceil n/\ell \rceil$ 個のブロック $x_1, \ldots, x_s \in \mathrm{GF}(2^\ell)$ に分ける（最後のブロックは 0 で埋める）
2. $t = 2\ell$ ビットのシードの前半 $\ell$ ビットを $\alpha$、後半 $\ell$ ビットを $\beta$ とする
3. 出力は、多項式の値と $\beta$ の内積の偶奇
-->

1. Split the input $x$ into $s = \lceil n/\ell \rceil$ blocks of $\ell$ bits, $x_1, \ldots, x_s \in \mathrm{GF}(2^\ell)$ (the last block is padded with 0s)
2. Let the first $\ell$ bits of the $t = 2\ell$-bit seed be $\alpha$ and the last $\ell$ bits be $\beta$
3. The output is the parity of the inner product of the polynomial's value and $\beta$

$$
p_\alpha(x) = \sum_{i=1}^{s} x_i\, \alpha^{s-i}, \qquad
C(x, (\alpha, \beta)) = \bigoplus_{j=0}^{\ell-1} \beta_j \cdot p_\alpha(x)_j
$$

<!-- 1 ビットあたりの誤差を $\varepsilon_1$ とすると、$\ell = \lceil \log_2 n + 2 \log_2 (2/\varepsilon_1) \rceil$ で、必要な min-entropy は $k_1 = 4\log_2(1/\varepsilon_1) + 6$ です（論文の Theorem B.7）。 -->

With an error of $\varepsilon_1$ per bit, $\ell = \lceil \log_2 n + 2 \log_2 (2/\varepsilon_1) \rceil$, and the required min-entropy is $k_1 = 4\log_2(1/\varepsilon_1) + 6$ (Theorem B.7 of the paper).

### C. weak design

<!-- **定義**（論文の Definition III.1）：$S_1, \ldots, S_m$ が weak $(m, t, r, d)$-design であるとは、すべての $i$ について -->

**Definition** (Definition III.1 of the paper): $S_1, \ldots, S_m$ is a weak $(m, t, r, d)$-design if, for every $i$,

$$
|S_i| = t, \qquad \sum_{j=1}^{i-1} 2^{|S_j \cap S_i|} \le r\,m
$$

<!-- が成り立つことです。 -->

holds.

<!-- **多項式による基本の design**（論文の Sec. III B 1）：$\mathrm{GF}(q)$ 上の多項式 $p$ ごとに集合 $S_p = \{(u, p(u))\}$ を作り、点 $(u, v)$ をシードの位置 $u q + v$ に対応させます。$i$ 番目の集合には、$i$ を $q$ 進数で書いた各桁を係数とする多項式（1 の位が定数項）を使います。この design は $r < 2e$ を満たします（Lemma C.1）。 -->

**Basic design from polynomials** (Sec. III B 1 of the paper): for each polynomial $p$ over $\mathrm{GF}(q)$ we form the set $S_p = \{(u, p(u))\}$, and map the point $(u, v)$ to the seed position $u q + v$. For the $i$-th set we use the polynomial whose coefficients are the digits of $i$ written in base $q$ (the ones digit is the constant term). This design satisfies $r < 2e$ (Lemma C.1).

<!-- **ブロック weak design**（論文の Sec. III B 2、$r_0 = 2e$）：基本の design を $L + 1$ 段に分けて並べ、$r = 1$ にします（Lemma C.2）。 -->

**Block weak design** (Sec. III B 2 of the paper, $r_0 = 2e$): the basic design is arranged in $L + 1$ stages to make $r = 1$ (Lemma C.2).

<!--
1. 段数 $L$：$r_0 + (m - r_0)(1 - 1/r_0)^L \le q$ を満たす最小の $L \ge 1$（論文の式 (3)）
2. $i$ 段目までの集合の数の累積：$(m - r_0)\left(1 - (1 - 1/r_0)^{i+1}\right)$（論文の式 (4) の和）。これを区切りにして段 $0, \ldots, L-1$ の大きさ $m_0, \ldots, m_{L-1}$ を決め、残り $m_L = m - \sum_{i<L} m_i$ を最後の段にする
3. 段 $i$ の集合は、シードの位置 $i q^2$ から $(i+1)q^2 - 1$ に置く。シードの長さは $d = (L + 1) q^2$
-->

1. Number of stages $L$: the smallest $L \ge 1$ satisfying $r_0 + (m - r_0)(1 - 1/r_0)^L \le q$ (Eq. (3) of the paper)
2. Cumulative number of sets up to stage $i$: $(m - r_0)\left(1 - (1 - 1/r_0)^{i+1}\right)$ (the sum in Eq. (4) of the paper). Using these as boundaries, determine the sizes $m_0, \ldots, m_{L-1}$ of stages $0, \ldots, L-1$, and put the rest, $m_L = m - \sum_{i<L} m_i$, in the last stage
3. The sets of stage $i$ are placed at seed positions $i q^2$ to $(i+1)q^2 - 1$. The seed length is $d = (L + 1) q^2$

<!-- ### D. パラメータ -->

### D. Parameters

<!-- 入力 $n$ ビット、1 ビットあたりの min-entropy $h$（入力全体で $k = nh$）、全体の誤差 $\varepsilon$ から、次のように決めます。 -->

From the $n$ input bits, the min-entropy per bit $h$ ($k = nh$ for the whole input), and the total error $\varepsilon$, the parameters are determined as follows.

<!--
| 量 | 式 |
|---|---|
| 1 ビットあたりの誤差 | $\varepsilon_1 = \varepsilon / m$（全体の誤差は $m\varepsilon_1 = \varepsilon$） |
| 体の次数 | $\ell = \lceil \log_2(4nm^2/\varepsilon^2) \rceil$ |
| 1 ビット抽出器のシード | $t = 2\ell$ |
| design の体の大きさ | $q$ = $t$ 以上の最小の 2 のべき |
| 出力の長さ | $k \ge m + k_1 = m + 4\log_2(m/\varepsilon) + 6$ を満たす最大の $m$ |
| シードの長さ | $d = (L + 1) q^2$（$m \le q$ のときは $q^2$） |
-->

| Quantity | Formula |
|---|---|
| Error per bit | $\varepsilon_1 = \varepsilon / m$ (the total error is $m\varepsilon_1 = \varepsilon$) |
| Degree of the field | $\ell = \lceil \log_2(4nm^2/\varepsilon^2) \rceil$ |
| Seed of the one-bit extractor | $t = 2\ell$ |
| Size of the design's field | $q$ = the smallest power of 2 that is at least $t$ |
| Output length | The largest $m$ satisfying $k \ge m + k_1 = m + 4\log_2(m/\varepsilon) + 6$ |
| Seed length | $d = (L + 1) q^2$ ($q^2$ when $m \le q$) |

<!-- ### 但し書き -->

### Caveats

<!--
- **$h$ の決め方**：$h$ は SP 800-90B の評価（`SP800_90B_textbook.ipynb`）で求めた値を `H_MIN_OVERRIDE` に入れると、さらに本番に近い実験検証になります。今回のノートブックでは Trevisan extractor のプログラム動作の確認のみを目的とするため、特に $h$ の値を指定しない場合は、最も基本的な評価手法である MCV 推定（2.）を用います。
- **シード**：raw data とは独立な一様乱数でなければなりません。`SEED_RNG` に整数を指定すると、誰でも再現できる疑似乱数になるので、確認用にだけ使ってください。
- **論文との違い**：このノートブックの実装には、論文の記述と違う点が 3 つあります。どれも weak design の条件を崩しません。
-->

- **How to choose $h$**: Setting `H_MIN_OVERRIDE` to the value obtained from the SP 800-90B assessment (`SP800_90B_textbook.ipynb`) makes the experiment closer to a real deployment. Because this notebook is intended only to check that the Trevisan extractor program works, the most basic assessment method, the MCV estimate (2.), is used when no value of $h$ is specified.
- **Seed**: It must be uniform random bits independent of the raw data. Setting `SEED_RNG` to an integer gives pseudorandom numbers that anyone can reproduce, so use it only for checking.
- **Differences from the paper**: The implementation in this notebook differs from the paper's description in three ways. None of them breaks the conditions of the weak design.

<!--
| 項目 | 論文 | このノートブック | 条件が崩れない理由 |
|---|---|---|---|
| design の体 | 大きさ $t$ の体（$t$ は素数のべき） | $q$ = $t$ 以上の最小の 2 のべきの $\mathrm{GF}(q)$ で、$u = 0, \ldots, t-1$ の $t$ 点だけを使う | 集合を一部の点に制限しても、重なりは増えない |
| $m \le q$ のとき | 式 (3) から $L = 1$ | 1 段だけ、$d = q^2$ | 定数の多項式の集合は互いに重ならない |
| 段の区切り | 式 (4) の累積 | 累積を切り上げた整数。$2e$ は有理数の上限・下限で扱い、両方で同じ値になることを確かめてから使う | 切り上げても Lemma C.2 の証明の 2 つの条件（$r_0 m_i \le m - \sum_{j<i} m_j$ と $m_L \le q$）が成り立つ |
-->

| Item | Paper | This notebook | Why the conditions still hold |
|---|---|---|---|
| Field of the design | A field of size $t$ ($t$ a prime power) | $\mathrm{GF}(q)$ with $q$ = the smallest power of 2 that is at least $t$, using only the $t$ points $u = 0, \ldots, t-1$ | Restricting the sets to some of the points does not increase their overlaps |
| When $m \le q$ | $L = 1$ from Eq. (3) | A single stage, $d = q^2$ | The sets of constant polynomials do not overlap each other |
| Stage boundaries | Cumulative sums of Eq. (4) | The cumulative sums rounded up to integers. $2e$ is handled with rational upper and lower bounds, and the result is used only after checking that both give the same value | Even with rounding up, the two conditions in the proof of Lemma C.2 ($r_0 m_i \le m - \sum_{j<i} m_j$ and $m_L \le q$) hold |

<!-- - **計算時間**：Python 版（7.1）は、出力 1 ビットごとに $\mathrm{GF}(2^\ell)$ の掛け算を $s = \lceil n/\ell \rceil$ 回行うので、計算時間は $n^2$ にほぼ比例し、$n = 10^4$ で数十秒かかります。大きい入力には `numba` を使う高速版（7.2）を使ってください。 -->

- **Computation time**: The Python version (7.1) performs $s = \lceil n/\ell \rceil$ multiplications in $\mathrm{GF}(2^\ell)$ for each output bit, so its time grows roughly as $n^2$ and takes several tens of seconds for $n = 10^4$. For large inputs, use the fast version with `numba` (7.2).

---

<!-- ## 使い方 -->

## How to use this notebook

<!-- 上のセルから順に実行してください。条件は、次の変数を書き換えて選びます。 -->

Run the cells in order from the top. The conditions are selected by editing the following variables.

<!--
| 変数 | 意味 | 既定値 |
|---|---|---|
| `N_BITS` | シミュレーションで作る入力のビット数 $n$ | `10_000` |
| `P1_GIVEN_0`, `P0_GIVEN_1` | シミュレーションの読み出し誤差 $P(1 \mid 0)$, $P(0 \mid 1)$ | `0.02`, `0.04` |
| `SIM_SEED` | シミュレーションの乱数シード（`None` なら毎回違う結果） | `1` |
| `INPUT_PATH` | raw data のファイル（`None` ならシミュレーションを使う） | `None` |
| `H_MIN_OVERRIDE` | 1 ビットあたりの min-entropy $h$（`None` なら MCV 推定値） | `None` |
| `EPSILON` | 全体の誤差 $\varepsilon$ | `2.0 ** -100` |
| `SEED_RNG` | シードの作り方（`None` なら OS の暗号用乱数、整数なら再現できる疑似乱数） | `None` |
| `USE_NUMBA` | 7.2 の高速版で抽出するか（`numba` がなければ `False` になる） | `True` |
| `SAVE_RESULTS` | 結果を保存するか（`True` なら `results/trevisan/<日時>/` に保存） | `False` |
-->

| Variable | Meaning | Default |
|---|---|---|
| `N_BITS` | Number of input bits $n$ generated by the simulation | `10_000` |
| `P1_GIVEN_0`, `P0_GIVEN_1` | Readout errors of the simulation $P(1 \mid 0)$, $P(0 \mid 1)$ | `0.02`, `0.04` |
| `SIM_SEED` | Random seed of the simulation (`None` gives different results each time) | `1` |
| `INPUT_PATH` | Raw-data file (`None` means use the simulation) | `None` |
| `H_MIN_OVERRIDE` | Min-entropy per bit $h$ (`None` means the MCV estimate) | `None` |
| `EPSILON` | Total error $\varepsilon$ | `2.0 ** -100` |
| `SEED_RNG` | How the seed is generated (`None`: the OS's cryptographic random numbers; integer: reproducible pseudorandom numbers) | `None` |
| `USE_NUMBA` | Whether to extract with the fast version in 7.2 (becomes `False` if `numba` is not available) | `True` |
| `SAVE_RESULTS` | Whether to save the results (`True` saves them in `results/trevisan/<date_time>/`) | `False` |

<!-- ### 保存されるファイル -->

### Saved files

<!-- 1. の準備のセルで `SAVE_RESULTS = True` にすると、そのセルを実行した日時の名前のフォルダ `results/trevisan/<年-月-日_時分秒>/`（変数 `RUN_DIR`）を作り、8. のセルで次のファイルを保存します。`False`（既定）では何も保存しません。 -->

If you set `SAVE_RESULTS = True` in the setup cell in 1., a folder named after the date and time that cell was run, `results/trevisan/<year-month-day_hourminutesecond>/` (variable `RUN_DIR`), is created, and the following files are saved in it by the cell in 8. With `False` (the default), nothing is saved.

<!--
```
results/trevisan/
└── 2026-11-10_153012/          RUN_DIR（1. のセルを実行した日時）
    ├── input_bits.txt             抽出の入力のビット列
    ├── trevisan_extracted.txt      抽出したビット列
    ├── trevisan_seed.txt           シード
    └── trevisan_info.json          パラメータと入力の情報
```
-->

```
results/trevisan/
└── 2026-11-10_153012/          RUN_DIR (date and time the cell in 1. was run)
    ├── input_bits.txt             input bit string of the extraction
    ├── trevisan_extracted.txt      extracted bit string
    ├── trevisan_seed.txt           seed
    └── trevisan_info.json          parameters and information about the input
```

<!--
| ファイル | 形式 | 中身 |
|---|---|---|
| `input_bits.txt` | テキスト。`'0'` と `'1'` の文字を改行なしで並べる（1 文字 = 1 ビット） | 抽出に使った入力のビット列（シミュレーションでも `INPUT_PATH` のファイルでも保存する） |
| `trevisan_extracted.txt` | 同じ形式 | 抽出したビット列。`load_bits` で読める。SP800_22 のノートブックの `BITS_FILE` に指定できる |
| `trevisan_seed.txt` | 同じ形式 | Trevisan's extractor のシード（$d$ ビット） |
| `trevisan_info.json` | JSON | 7.3 の `extraction_info` の内容（6. の表の `plan_trevisan` のキーと、`irreducible_polynomial`, `design_irreducible_polynomial`, `seed_source`, `extraction_s`）に、`input_path`, `simulation`, `h_min_source` を加えたもの。`input_path` は `INPUT_PATH` の値（シミュレーションなら `null`）、`simulation` はシミュレーションの条件（`N_BITS`, `P1_GIVEN_0`, `P0_GIVEN_1`, `SIM_SEED`）、`h_min_source` は `"MCV"` か `"H_MIN_OVERRIDE"` |
-->

| File | Format | Contents |
|---|---|---|
| `input_bits.txt` | Text. The characters `'0'` and `'1'` with no line breaks (1 character = 1 bit) | The input bit string of the extraction (saved both for the simulation and for an `INPUT_PATH` file) |
| `trevisan_extracted.txt` | Same format | The extracted bit string. Can be read with `load_bits`, and can be given as `BITS_FILE` in the SP800_22 notebook |
| `trevisan_seed.txt` | Same format | The seed of Trevisan's extractor ($d$ bits) |
| `trevisan_info.json` | JSON | The contents of `extraction_info` from 7.3 (the keys of `plan_trevisan` in the table in 6., plus `irreducible_polynomial`, `design_irreducible_polynomial`, `seed_source`, and `extraction_s`), plus `input_path`, `simulation`, and `h_min_source`. `input_path` is the value of `INPUT_PATH` (`null` for the simulation), `simulation` holds the simulation conditions (`N_BITS`, `P1_GIVEN_0`, `P0_GIVEN_1`, `SIM_SEED`), and `h_min_source` is `"MCV"` or `"H_MIN_OVERRIDE"` |

<!--
- ファイルは上書きしません。同じ `RUN_DIR` のまま 8. のセルをもう一度実行するとエラー（`FileExistsError`）になるので、1. の準備のセルを実行し直して新しいフォルダを作ってください。
- 入力・シード・パラメータがそろっているので、`input_bits.txt` と `trevisan_seed.txt` を同じ設定で抽出の関数に渡すと、同じ出力が得られます。
-->

- Files are never overwritten. If you run the cell in 8. again with the same `RUN_DIR`, an error (`FileExistsError`) is raised, so rerun the setup cell in 1. to create a new folder.
- Because the input, the seed, and the parameters are all saved, passing `input_bits.txt` and `trevisan_seed.txt` to the extraction function with the same settings gives the same output.

<!-- ### 関数の家系図 -->

### Function call tree

<!-- このノートブックの関数の呼び出し関係です。上の（字下げの浅い）関数が、そのすぐ下に枝で続く関数を内部で呼び出します。 -->

This shows how the functions in this notebook call each other. A function higher up (less indented) internally calls the functions on the branches directly below it.

<!--
- `[5.]` のような右の番号は、**その関数が定義されている章**です。
- `← 7.3 のセルから実行` は、その章のコードセルから直接呼び出していることを示します。
- `（↑ 上で展開済み）` は、同じ関数の中身をすでに上の図で展開していることを示します。
-->

- The number on the right, such as `[5.]`, is **the section in which the function is defined**.
- `← run from the cell in 7.3` means the function is called directly from the code cell of that section.
- `(↑ expanded above)` means the contents of the same function have already been expanded earlier in the tree.

<!-- **抽出の処理** -->

**Extraction**

<!--
```
trevisan_extraction                       [7.1]  ← 7.3 のセルから実行
├── _prepare_extraction                   [7.1]
│   ├── binary_vector                     [1.]
│   ├── plan_trevisan                     [6.]  ← 6. のセルから実行
│   │   ├── positive_integer              [1.]
│   │   ├── required_min_entropy          [6.]
│   │   ├── field_degree                  [6.]
│   │   └── block_sizes                   [5.]
│   │       ├── positive_integer          [1.]
│   │       └── _e_bounds                 [5.]
│   ├── irreducible_polynomial            [3.]
│   │   └── is_irreducible                [3.]
│   │       ├── polynomial_remainder      [3.]
│   │       ├── gf_multiply               [3.]
│   │       │   └── polynomial_remainder  [3.]
│   │       └── polynomial_gcd            [3.]
│   │           └── polynomial_remainder  [3.]
│   └── generate_seed                     [7.1]
├── rs_coefficients                       [4.]
│   ├── binary_vector                     [1.]
│   ├── positive_integer                  [1.]
│   └── bits_to_integer                   [3.]
├── design_positions                      [5.]
│   ├── block_position                    [5.]
│   └── weak_design_set                   [5.]
│       └── evaluate_polynomial           [3.]
│           └── gf_multiply               [3.]  （↑ 上で展開済み）
└── one_bit_extractor                     [4.]
    ├── binary_vector                     [1.]
    ├── bits_to_integer                   [3.]
    └── evaluate_polynomial               [3.]  （↑ 上で展開済み）

trevisan_extraction_numba                 [7.2]  ← 7.3 のセルから実行
├── _prepare_extraction                   [7.1]  （↑ 上で展開済み）
├── to_limbs                              [7.2]
├── rs_coefficients                       [4.]  （↑ 上で展開済み）
├── block_position                        [5.]
├── design_set_numba                      [7.2]
├── bits_to_integer                       [3.]
└── rs_hadamard_numba                     [7.2]
```
-->

```
trevisan_extraction                       [7.1]  ← run from the cell in 7.3
├── _prepare_extraction                   [7.1]
│   ├── binary_vector                     [1.]
│   ├── plan_trevisan                     [6.]  ← run from the cell in 6.
│   │   ├── positive_integer              [1.]
│   │   ├── required_min_entropy          [6.]
│   │   ├── field_degree                  [6.]
│   │   └── block_sizes                   [5.]
│   │       ├── positive_integer          [1.]
│   │       └── _e_bounds                 [5.]
│   ├── irreducible_polynomial            [3.]
│   │   └── is_irreducible                [3.]
│   │       ├── polynomial_remainder      [3.]
│   │       ├── gf_multiply               [3.]
│   │       │   └── polynomial_remainder  [3.]
│   │       └── polynomial_gcd            [3.]
│   │           └── polynomial_remainder  [3.]
│   └── generate_seed                     [7.1]
├── rs_coefficients                       [4.]
│   ├── binary_vector                     [1.]
│   ├── positive_integer                  [1.]
│   └── bits_to_integer                   [3.]
├── design_positions                      [5.]
│   ├── block_position                    [5.]
│   └── weak_design_set                   [5.]
│       └── evaluate_polynomial           [3.]
│           └── gf_multiply               [3.]  (↑ expanded above)
└── one_bit_extractor                     [4.]
    ├── binary_vector                     [1.]
    ├── bits_to_integer                   [3.]
    └── evaluate_polynomial               [3.]  (↑ expanded above)

trevisan_extraction_numba                 [7.2]  ← run from the cell in 7.3
├── _prepare_extraction                   [7.1]  (↑ expanded above)
├── to_limbs                              [7.2]
├── rs_coefficients                       [4.]  (↑ expanded above)
├── block_position                        [5.]
├── design_set_numba                      [7.2]
├── bits_to_integer                       [3.]
└── rs_hadamard_numba                     [7.2]
```

<!-- **入力の準備・評価・保存（他の関数を呼び出さず、セルから単独で使う関数）** -->

**Preparing, assessing, and saving the input (functions used on their own from a cell, without calling other functions)**

<!--
```
simulate_c1_bits  [2.]  ← 2. のセルから実行
load_bits         [2.]  ← 2. のセルから実行
mcv_min_entropy   [2.]  ← 2. のセルから実行
save_bits         [2.]  ← 8. のセルから実行
```
-->

```
simulate_c1_bits  [2.]  ← run from the cell in 2.
load_bits         [2.]  ← run from the cell in 2.
mcv_min_entropy   [2.]  ← run from the cell in 2.
save_bits         [2.]  ← run from the cell in 8.
```

<!-- **章ごとの定義の一覧** -->

**Functions defined in each section**

<!--
| 章 | 章の題 | 定義されている関数 |
|---|---|---|
| 1. | 準備 | `binary_vector`、`positive_integer` |
| 2. | 入力のビット列を用意する | `simulate_c1_bits`、`load_bits`、`save_bits`、`mcv_min_entropy` |
| 3. | 有限体 $\mathrm{GF}(2^\ell)$ の演算 | `polynomial_remainder`、`gf_multiply`、`polynomial_gcd`、`is_irreducible`、`irreducible_polynomial`、`evaluate_polynomial`、`bits_to_integer` |
| 4. | 1 ビット抽出器（RSH） | `rs_coefficients`、`one_bit_extractor` |
| 5. | weak design | `weak_design_set`、`_e_bounds`、`block_sizes`、`block_position`、`design_positions` |
| 6. | パラメータを決める | `required_min_entropy`、`field_degree`、`plan_trevisan` |
| 7.1 | 抽出の関数（Python 版） | `generate_seed`、`_prepare_extraction`、`trevisan_extraction` |
| 7.2 | 抽出の関数（`numba` を使う高速版） | `to_limbs`、`design_set_numba`、`rs_hadamard_numba`、`trevisan_extraction_numba` |
-->

| Section | Section title | Functions defined |
|---|---|---|
| 1. | Setup | `binary_vector`, `positive_integer` |
| 2. | Preparing the input bit string | `simulate_c1_bits`, `load_bits`, `save_bits`, `mcv_min_entropy` |
| 3. | Arithmetic in the finite field $\mathrm{GF}(2^\ell)$ | `polynomial_remainder`, `gf_multiply`, `polynomial_gcd`, `is_irreducible`, `irreducible_polynomial`, `evaluate_polynomial`, `bits_to_integer` |
| 4. | One-bit extractor (RSH) | `rs_coefficients`, `one_bit_extractor` |
| 5. | Weak design | `weak_design_set`, `_e_bounds`, `block_sizes`, `block_position`, `design_positions` |
| 6. | Determining the parameters | `required_min_entropy`, `field_degree`, `plan_trevisan` |
| 7.1 | Extraction functions (Python version) | `generate_seed`, `_prepare_extraction`, `trevisan_extraction` |
| 7.2 | Extraction functions (fast version with `numba`) | `to_limbs`, `design_set_numba`, `rs_hadamard_numba`, `trevisan_extraction_numba` |

<!-- ## 1. 準備 -->

## 1. Setup

<!-- ライブラリを読み込み、全体の誤差 `EPSILON` とシードの作り方 `SEED_RNG` を決めます。`numba` があれば高速版（7.2）を使えるようにします。 -->

We load the libraries and set the total error `EPSILON` and how the seed is generated, `SEED_RNG`. If `numba` is available, the fast version (7.2) is enabled.

<!-- 結果の保存：`SAVE_RESULTS` で、8. で入力のビット列と抽出結果を保存するかどうかを選びます。`True` にすると、このセルを実行した日時の名前のフォルダ（例：`results/trevisan/2026-11-10_153012/`）に保存します。`False`（既定）では何も保存しません。 -->

Saving the results: `SAVE_RESULTS` selects whether the input bit string and the extraction results are saved in 8. With `True`, they are saved in a folder named after the date and time this cell was run (e.g., `results/trevisan/2026-11-10_153012/`). With `False` (the default), nothing is saved.

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 目的 | 戻り値・処理 | 使用例 |
|---|---|---|---|
| `binary_vector` | 入力が 0 と 1 だけからなることを検査し、`np.uint8` の 1 次元配列にそろえることを目的とした関数です。 | 0 と 1 の `np.uint8` の配列。それ以外の値があればエラー | `binary_vector([0, 1, 1])` |
| `positive_integer` | 値が 1 以上の整数であることを検査し、Python の int にそろえることを目的とした関数です。 | int。1 以上の整数でなければエラー | `positive_integer(5, "m")` |
-->

| Function | Purpose | Return value / behavior | Example |
|---|---|---|---|
| `binary_vector` | Checks that the input consists only of 0s and 1s and converts it to a 1-D `np.uint8` array. | `np.uint8` array of 0s and 1s. An error is raised for any other value | `binary_vector([0, 1, 1])` |
| `positive_integer` | Checks that a value is an integer of at least 1 and converts it to a Python int. | int. An error is raised if it is not an integer of at least 1 | `positive_integer(5, "m")` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td rowspan="2"><code>binary_vector</code></td><td><code>values</code></td><td>0 と 1 からなる配列・リスト</td><td>必須</td></tr>
<tr><td><code>name</code></td><td>エラーメッセージに使う名前</td><td><code>"raw_bits"</code></td></tr>
<tr><td rowspan="2"><code>positive_integer</code></td><td><code>value</code></td><td>1 以上の整数</td><td>必須</td></tr>
<tr><td><code>name</code></td><td>エラーメッセージに使う名前</td><td>必須</td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td rowspan="2"><code>binary_vector</code></td><td><code>values</code></td><td>Array or list of 0s and 1s</td><td>Required</td></tr>
<tr><td><code>name</code></td><td>Name used in error messages</td><td><code>"raw_bits"</code></td></tr>
<tr><td rowspan="2"><code>positive_integer</code></td><td><code>value</code></td><td>Integer of at least 1</td><td>Required</td></tr>
<tr><td><code>name</code></td><td>Name used in error messages</td><td>Required</td></tr>
</tbody>
</table>

In [ ]:
import json
import math
import secrets
import time
from bisect import bisect_right
from fractions import Fraction
from functools import lru_cache
from datetime import datetime
from pathlib import Path

import numpy as np

# Total error
EPSILON = 2.0 ** -100

# How the seed is generated (None: the OS's cryptographic random numbers; integer: reproducible pseudorandom numbers)
SEED_RNG = None

# True: save the files of this run in a new folder results/trevisan/<date_time>/; False: save nothing
SAVE_RESULTS = False
RESULTS_ROOT = Path("results/trevisan")
RUN_DIR = RESULTS_ROOT / datetime.now().strftime("%Y-%m-%d_%H%M%S") if SAVE_RESULTS else None
print("save to    :", RUN_DIR if RUN_DIR is not None else "nothing is saved (SAVE_RESULTS = False)")

try:
    from numba import njit
    NUMBA_AVAILABLE = True
except ImportError:
    NUMBA_AVAILABLE = False
print("numba:", "available (the fast version in 7.2 can be used)" if NUMBA_AVAILABLE else "not available (only the Python version in 7.1 is used)")


def binary_vector(values, name="raw_bits"):
    a = np.asarray(values).reshape(-1)
    if a.size == 0:
        raise ValueError(f"{name} is empty")
    if not np.all((a == 0) | (a == 1)):
        raise ValueError(f"{name} must contain only 0 and 1")
    return a.astype(np.uint8)


def positive_integer(value, name):
    if isinstance(value, bool) or not isinstance(value, (int, np.integer)) or value < 1:
        raise ValueError(f"{name} must be a positive integer")
    return int(value)

<!-- ## 2. 入力のビット列を用意する -->

## 2. Preparing the input bit string

<!-- C1 回路（1 量子ビットの重ね合わせ状態の測定）の出力を模したビット列を作ります。公平なビットに非対称な読み出し誤差 $P(1|0)$, $P(0|1)$ を加えたもので、1 の割合は $p_1 = \frac{1}{2}(1 - P(0|1)) + \frac{1}{2}P(1|0)$ です。`INPUT_PATH` を指定すると、そのファイルの raw data を使います。 -->

We create a bit string that mimics the output of the C1 circuit (measurement of a 1-qubit superposition). It consists of fair bits with asymmetric readout errors $P(1|0)$, $P(0|1)$ added, so the proportion of 1s is $p_1 = \frac{1}{2}(1 - P(0|1)) + \frac{1}{2}P(1|0)$. If `INPUT_PATH` is set, the raw data in that file is used.

<!-- 1 ビットあたりの min-entropy $h$ は、`H_MIN_OVERRIDE` を指定しない場合、SP 800-90B の Most Common Value（MCV）推定で決めます。 -->

If `H_MIN_OVERRIDE` is not set, the min-entropy per bit $h$ is determined by the Most Common Value (MCV) estimate of SP 800-90B.

$$
p_u = \min\left(1,\ \hat{p} + 2.576\sqrt{\frac{\hat{p}(1-\hat{p})}{L-1}}\right), \qquad h = -\log_2 p_u
$$

<!-- $\hat{p}$ は $L$ 個のサンプルのうち最も多い値の割合です。 -->

$\hat{p}$ is the proportion of the most common value among the $L$ samples.

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 目的 | 戻り値・処理 | 使用例 |
|---|---|---|---|
| `simulate_c1_bits` | 実機の raw data がないときの入力として、公平なビットに読み出し誤差によるビット反転を加え、C1 回路の測定結果を模擬したビット列を作ることを目的とした関数です。 | 長さ `n_bits` のビット列（`np.uint8` の配列） | `simulate_c1_bits(10, 0.02, 0.04, rng=1)` |
| `load_bits` | raw data などの 0 と 1 のテキストファイルを読み込み、抽出の入力に使える配列に変換することを目的とした関数です。 | ファイルのビット列（`np.uint8` の配列） | `load_bits("results/c1/2026-11-10_153012/4.5_ideal_trials/c1_H_ideal_trial001.txt")` |
| `save_bits` | ビット列を 0 と 1 の文字（改行なし）のテキストファイルとして、既存のファイルを上書きせずに保存することを目的とした関数です。 | 保存したパス | `save_bits("out/bits.txt", bits)` |
| `mcv_min_entropy` | 最頻値の出現割合の 99% 信頼上限 $p_u$ から、1 サンプルあたりの min-entropy $-\log_2 p_u$ を推定（MCV 推定）することを目的とした関数です。 | MCV 推定による 1 サンプルあたりの min-entropy（float） | `mcv_min_entropy(raw_bits)` |
-->

| Function | Purpose | Return value / behavior | Example |
|---|---|---|---|
| `simulate_c1_bits` | Creates a bit string that mimics the measurement results of the C1 circuit by adding bit flips from readout errors to fair bits, as input when no raw data from real hardware is available. | Bit string of length `n_bits` (`np.uint8` array) | `simulate_c1_bits(10, 0.02, 0.04, rng=1)` |
| `load_bits` | Reads a text file of 0s and 1s, such as raw data, and converts it into an array that can be used as input for extraction. | Bit string of the file (`np.uint8` array) | `load_bits("results/c1/2026-11-10_153012/4.5_ideal_trials/c1_H_ideal_trial001.txt")` |
| `save_bits` | Saves a bit string as a text file of the characters 0 and 1 (no newlines), without overwriting an existing file. | Path of the saved file | `save_bits("out/bits.txt", bits)` |
| `mcv_min_entropy` | Estimates the min-entropy per sample $-\log_2 p_u$ from the 99% upper confidence bound $p_u$ of the proportion of the most common value (the MCV estimate). | Min-entropy per sample from the MCV estimate (float) | `mcv_min_entropy(raw_bits)` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td rowspan="4"><code>simulate_c1_bits</code></td><td><code>n_bits</code></td><td>作るビット数</td><td>必須</td></tr>
<tr><td><code>p1_given_0</code></td><td>$P(1|0)$</td><td><code>0.0</code></td></tr>
<tr><td><code>p0_given_1</code></td><td>$P(0|1)$</td><td><code>0.0</code></td></tr>
<tr><td><code>rng</code></td><td>乱数シード（整数）、<code>np.random.Generator</code>、または <code>None</code></td><td><code>None</code></td></tr>
<tr><td><code>load_bits</code></td><td><code>path</code></td><td>0 と 1 の文字だけのテキストファイル（空白・改行は無視）</td><td>必須</td></tr>
<tr><td rowspan="2"><code>save_bits</code></td><td><code>path</code></td><td>保存先。既に存在する場合はエラー</td><td>必須</td></tr>
<tr><td><code>bits</code></td><td>0 と 1 の配列</td><td>必須</td></tr>
<tr><td><code>mcv_min_entropy</code></td><td><code>symbols</code></td><td>整数の配列。長さ 2 以上</td><td>必須</td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td rowspan="4"><code>simulate_c1_bits</code></td><td><code>n_bits</code></td><td>Number of bits to generate</td><td>Required</td></tr>
<tr><td><code>p1_given_0</code></td><td>$P(1|0)$</td><td><code>0.0</code></td></tr>
<tr><td><code>p0_given_1</code></td><td>$P(0|1)$</td><td><code>0.0</code></td></tr>
<tr><td><code>rng</code></td><td>Random seed (integer), <code>np.random.Generator</code>, or <code>None</code></td><td><code>None</code></td></tr>
<tr><td><code>load_bits</code></td><td><code>path</code></td><td>Text file of only the characters 0 and 1 (whitespace and newlines are ignored)</td><td>Required</td></tr>
<tr><td rowspan="2"><code>save_bits</code></td><td><code>path</code></td><td>Destination. An error is raised if it already exists</td><td>Required</td></tr>
<tr><td><code>bits</code></td><td>Array of 0s and 1s</td><td>Required</td></tr>
<tr><td><code>mcv_min_entropy</code></td><td><code>symbols</code></td><td>Array of integers. Length 2 or more</td><td>Required</td></tr>
</tbody>
</table>

In [ ]:
def simulate_c1_bits(n_bits, p1_given_0=0.0, p0_given_1=0.0, rng=None):
    rng = np.random.default_rng(rng)
    ideal = rng.integers(0, 2, size=n_bits, dtype=np.uint8)        # ideal measurement results: fair bits
    u = rng.random(n_bits)
    flip = np.where(ideal == 0, u < p1_given_0, u < p0_given_1)    # flipped by readout errors
    return ideal ^ flip.astype(np.uint8)


def load_bits(path):
    text = "".join(Path(path).read_text(encoding="ascii").split())
    bits = np.frombuffer(text.encode("ascii"), dtype=np.uint8) - ord("0")
    if bits.size == 0 or np.any(bits > 1):
        raise ValueError(f"the file contains characters other than 0 and 1, or is empty: {path}")
    return bits.astype(np.uint8)


def save_bits(path, bits):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    bits = np.asarray(bits, dtype=np.uint8).reshape(-1)
    with path.open("x", encoding="ascii") as f:
        f.write((bits + ord("0")).tobytes().decode("ascii"))
    return path


def mcv_min_entropy(symbols):
    symbols = np.asarray(symbols).reshape(-1)
    L = len(symbols)
    _, counts = np.unique(symbols, return_counts=True)
    p_hat = counts.max() / L
    p_u = min(1.0, p_hat + 2.576 * math.sqrt(p_hat * (1 - p_hat) / (L - 1)))
    return -math.log2(p_u)

In [ ]:
N_BITS = 10_000
P1_GIVEN_0 = 0.02
P0_GIVEN_1 = 0.04
SIM_SEED = 1
INPUT_PATH = None       # e.g., Path("results/c1/2026-11-10_153012/4.5_ideal_trials/c1_H_ideal_trial001.txt")
H_MIN_OVERRIDE = None   # e.g., the result of the SP 800-90B assessment

raw_bits = simulate_c1_bits(N_BITS, P1_GIVEN_0, P0_GIVEN_1, rng=SIM_SEED) if INPUT_PATH is None else load_bits(INPUT_PATH)
h_min = mcv_min_entropy(raw_bits) if H_MIN_OVERRIDE is None else float(H_MIN_OVERRIDE)
print(f"Input: n = {raw_bits.size}, P(1) = {raw_bits.mean():.5f}")
print(f"h used for extraction = {h_min:.6f} ({'MCV estimate' if H_MIN_OVERRIDE is None else 'H_MIN_OVERRIDE'})")

<!-- ## 3. 有限体 $\mathrm{GF}(2^\ell)$ の演算 -->

## 3. Arithmetic in the finite field $\mathrm{GF}(2^\ell)$

<!-- 手法 B の $\mathrm{GF}(2^\ell)$ の演算を、Python の整数で実装します。整数のビット $j$ が $X^j$ の係数です（たとえば $X^8 + X^4 + X^3 + X + 1$ は `0x11b`）。 -->

We implement the $\mathrm{GF}(2^\ell)$ arithmetic of Method B with Python integers. Bit $j$ of an integer is the coefficient of $X^j$ (for example, $X^8 + X^4 + X^3 + X + 1$ is `0x11b`).

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 目的 | 戻り値・処理 | 使用例 |
|---|---|---|---|
| `polynomial_remainder` | int のビットを係数とする $\mathrm{GF}(2)$ 上の多項式について、割り算の余りを計算することを目的とした関数です。 | 多項式 `a` を `modulus` で割った余り（int） | `polynomial_remainder(0b10011, 0b1011)` |
| `gf_multiply` | $\mathrm{GF}(2^\ell)$ の要素どうしの掛け算（多項式の積を既約多項式 $f$ で割った余り）を計算することを目的とした関数です。 | 積 $ab \bmod f$（int） | `gf_multiply(0x57, 0x83, 0x11b)` |
| `polynomial_gcd` | 既約判定に使うため、$\mathrm{GF}(2)$ 上の 2 つの多項式の最大公約多項式をユークリッドの互除法で計算することを目的とした関数です。 | 最大公約多項式（int） | `polynomial_gcd(0b110, 0b11)` |
| `is_irreducible` | 多項式が $\mathrm{GF}(2)$ 上で既約かどうかを、Rabin の判定法で判定することを目的とした関数です。 | Rabin の判定法で既約なら `True` | `is_irreducible(0x11b)` |
| `irreducible_polynomial` | 有限体 $\mathrm{GF}(2^\ell)$ の定義に使う $\ell$ 次の既約多項式を、候補を小さい順に調べて一意に求めることを目的とした関数です。 | 定数項が 1 の最初の $\ell$ 次既約多項式（int） | `irreducible_polynomial(8)` |
| `evaluate_polynomial` | 有限体上の多項式の、点 $\alpha$ での値を Horner 法で計算することを目的とした関数です。 | Horner 法で計算した $\sum_i c_i \alpha^i$（int） | `evaluate_polynomial([1, 2, 3], 5, 0x11b)` |
| `bits_to_integer` | ビットの配列を、有限体の要素や多項式として扱える整数に変換することを目的とした関数です。 | ビット $j$ を $2^j$ の位とする整数（int） | `bits_to_integer([1, 0, 1])` |
-->

| Function | Purpose | Return value / behavior | Example |
|---|---|---|---|
| `polynomial_remainder` | Computes the remainder of division for polynomials over $\mathrm{GF}(2)$ whose coefficients are the bits of an int. | Remainder of the polynomial `a` divided by `modulus` (int) | `polynomial_remainder(0b10011, 0b1011)` |
| `gf_multiply` | Computes the product of two elements of $\mathrm{GF}(2^\ell)$ (the product of polynomials reduced modulo the irreducible polynomial $f$). | Product $ab \bmod f$ (int) | `gf_multiply(0x57, 0x83, 0x11b)` |
| `polynomial_gcd` | Computes the greatest common divisor of two polynomials over $\mathrm{GF}(2)$ with the Euclidean algorithm, for use in the irreducibility test. | Greatest common divisor polynomial (int) | `polynomial_gcd(0b110, 0b11)` |
| `is_irreducible` | Tests with Rabin's test whether a polynomial is irreducible over $\mathrm{GF}(2)$. | `True` if irreducible by Rabin's test | `is_irreducible(0x11b)` |
| `irreducible_polynomial` | Finds, uniquely, the degree-$\ell$ irreducible polynomial used to define the finite field $\mathrm{GF}(2^\ell)$ by checking the candidates in increasing order. | The first degree-$\ell$ irreducible polynomial with constant term 1 (int) | `irreducible_polynomial(8)` |
| `evaluate_polynomial` | Computes the value of a polynomial over the finite field at the point $\alpha$ with Horner's method. | $\sum_i c_i \alpha^i$ computed with Horner's method (int) | `evaluate_polynomial([1, 2, 3], 5, 0x11b)` |
| `bits_to_integer` | Converts an array of bits into an integer that can be treated as a finite-field element or a polynomial. | Integer in which bit $j$ is the $2^j$ place (int) | `bits_to_integer([1, 0, 1])` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td rowspan="2"><code>polynomial_remainder</code></td><td><code>a</code></td><td>割られる多項式（int）</td><td>必須</td></tr>
<tr><td><code>modulus</code></td><td>割る多項式（int、0 以外）</td><td>必須</td></tr>
<tr><td rowspan="3"><code>gf_multiply</code></td><td><code>a</code></td><td>体の要素（int）</td><td>必須</td></tr>
<tr><td><code>b</code></td><td>体の要素（int）</td><td>必須</td></tr>
<tr><td><code>modulus</code></td><td>既約多項式 $f$（int）</td><td>必須</td></tr>
<tr><td rowspan="2"><code>polynomial_gcd</code></td><td><code>a</code></td><td>多項式（int）</td><td>必須</td></tr>
<tr><td><code>b</code></td><td>多項式（int）</td><td>必須</td></tr>
<tr><td><code>is_irreducible</code></td><td><code>polynomial</code></td><td>多項式（int）</td><td>必須</td></tr>
<tr><td><code>irreducible_polynomial</code></td><td><code>ell</code></td><td>次数 $\ell$（1 以上の整数）</td><td>必須</td></tr>
<tr><td rowspan="3"><code>evaluate_polynomial</code></td><td><code>coefficients</code></td><td>係数のリスト（低次から順）</td><td>必須</td></tr>
<tr><td><code>alpha</code></td><td>代入する値</td><td>必須</td></tr>
<tr><td><code>modulus</code></td><td>既約多項式</td><td>必須</td></tr>
<tr><td><code>bits_to_integer</code></td><td><code>bits</code></td><td>0 と 1 の配列（先頭がビット 0）</td><td>必須</td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td rowspan="2"><code>polynomial_remainder</code></td><td><code>a</code></td><td>Dividend polynomial (int)</td><td>Required</td></tr>
<tr><td><code>modulus</code></td><td>Divisor polynomial (int, nonzero)</td><td>Required</td></tr>
<tr><td rowspan="3"><code>gf_multiply</code></td><td><code>a</code></td><td>Field element (int)</td><td>Required</td></tr>
<tr><td><code>b</code></td><td>Field element (int)</td><td>Required</td></tr>
<tr><td><code>modulus</code></td><td>Irreducible polynomial $f$ (int)</td><td>Required</td></tr>
<tr><td rowspan="2"><code>polynomial_gcd</code></td><td><code>a</code></td><td>Polynomial (int)</td><td>Required</td></tr>
<tr><td><code>b</code></td><td>Polynomial (int)</td><td>Required</td></tr>
<tr><td><code>is_irreducible</code></td><td><code>polynomial</code></td><td>Polynomial (int)</td><td>Required</td></tr>
<tr><td><code>irreducible_polynomial</code></td><td><code>ell</code></td><td>Degree $\ell$ (integer of at least 1)</td><td>Required</td></tr>
<tr><td rowspan="3"><code>evaluate_polynomial</code></td><td><code>coefficients</code></td><td>List of coefficients (from the lowest degree)</td><td>Required</td></tr>
<tr><td><code>alpha</code></td><td>Value to substitute</td><td>Required</td></tr>
<tr><td><code>modulus</code></td><td>Irreducible polynomial</td><td>Required</td></tr>
<tr><td><code>bits_to_integer</code></td><td><code>bits</code></td><td>Array of 0s and 1s (the first element is bit 0)</td><td>Required</td></tr>
</tbody>
</table>

In [ ]:
def polynomial_remainder(a, modulus):
    # shift modulus so that the highest-degree term cancels, and XOR (long division of polynomials)
    while a.bit_length() >= modulus.bit_length():
        a ^= modulus << (a.bit_length() - modulus.bit_length())
    return a


def gf_multiply(a, b, modulus):
    # carry-less multiplication: for each 1 bit of b, shift a and XOR
    product = 0
    while b:
        low = b & -b                     # lowest 1 bit of b
        product ^= a << (low.bit_length() - 1)
        b ^= low
    return polynomial_remainder(product, modulus)


def polynomial_gcd(a, b):
    while b:
        a, b = b, polynomial_remainder(a, b)
    return a


def is_irreducible(polynomial):
    ell = polynomial.bit_length() - 1
    if ell < 1 or not polynomial & 1:
        return False
    factors, remainder, p = set(), ell, 2       # prime factors p of ell
    while p * p <= remainder:
        while remainder % p == 0:
            factors.add(p)
            remainder //= p
        p += 1
    if remainder > 1:
        factors.add(remainder)
    checks = {ell // p for p in factors}
    x = polynomial_remainder(2, polynomial)     # the polynomial X
    power = x
    for i in range(1, ell + 1):
        power = gf_multiply(power, power, polynomial)   # X^(2^i) mod f
        if i in checks and polynomial_gcd(power ^ x, polynomial) != 1:
            return False
    return power == x


@lru_cache(maxsize=32)
def irreducible_polynomial(ell):
    if ell < 1:
        raise ValueError("field degree must be positive")
    for low in range(1, 1 << ell, 2):
        candidate = (1 << ell) | low
        if is_irreducible(candidate):
            return candidate
    raise RuntimeError("No irreducible polynomial found")


def evaluate_polynomial(coefficients, alpha, modulus):
    # Horner's method: (...((c_{s-1} α + c_{s-2}) α + c_{s-3}) ...) α + c_0
    value = 0
    for coefficient in reversed(coefficients):
        value = gf_multiply(value, alpha, modulus) ^ int(coefficient)
    return value


def bits_to_integer(bits):
    bits = np.asarray(bits, dtype=np.uint8).reshape(-1)
    return int.from_bytes(np.packbits(bits, bitorder="little").tobytes(), "little")

<!-- ## 4. 1 ビット抽出器（RSH） -->

## 4. One-bit extractor (RSH)

<!-- 手法 B の RSH を実装します。$p_\alpha(x) = \sum_{i=1}^{s} x_i \alpha^{s-i}$ では先頭のブロック $x_1$ が最高次なので、`rs_coefficients` は係数を低次から並べるためにブロックの並びを逆にします。 -->

We implement the RSH of Method B. In $p_\alpha(x) = \sum_{i=1}^{s} x_i \alpha^{s-i}$ the first block $x_1$ has the highest degree, so `rs_coefficients` reverses the order of the blocks to list the coefficients from the lowest degree.

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 目的 | 戻り値・処理 | 使用例 |
|---|---|---|---|
| `rs_coefficients` | 入力を $\ell$ ビットずつのブロックに分け、Reed–Solomon 符号の多項式 $p_\alpha$ の係数に変換することを目的とした関数です。 | 係数のリスト（低次から順、長さ $s = \lceil n/\ell \rceil$） | `rs_coefficients([1, 0, 1, 1, 0], 2)` |
| `one_bit_extractor` | 多項式の係数とシード $y_{S_i}$ から、RSH 1 ビット抽出器の出力 $C(x, y_{S_i})$ を計算することを目的とした関数です。 | 1 ビットの出力 $C(x, y_{S_i})$（0 または 1） | `one_bit_extractor(coefficients, sub_seed, ell, modulus)` |
-->

| Function | Purpose | Return value / behavior | Example |
|---|---|---|---|
| `rs_coefficients` | Splits the input into blocks of $\ell$ bits and converts them into the coefficients of the Reed–Solomon polynomial $p_\alpha$. | List of coefficients (from the lowest degree, length $s = \lceil n/\ell \rceil$) | `rs_coefficients([1, 0, 1, 1, 0], 2)` |
| `one_bit_extractor` | Computes the output $C(x, y_{S_i})$ of the RSH one-bit extractor from the polynomial's coefficients and the seed $y_{S_i}$. | One output bit $C(x, y_{S_i})$ (0 or 1) | `one_bit_extractor(coefficients, sub_seed, ell, modulus)` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td rowspan="2"><code>rs_coefficients</code></td><td><code>raw_bits</code></td><td>入力のビット列</td><td>必須</td></tr>
<tr><td><code>ell</code></td><td>体の次数 $\ell$</td><td>必須</td></tr>
<tr><td rowspan="4"><code>one_bit_extractor</code></td><td><code>coefficients</code></td><td><code>rs_coefficients</code> の戻り値</td><td>必須</td></tr>
<tr><td><code>sub_seed</code></td><td>長さ $2\ell$ のシード $y_{S_i}$</td><td>必須</td></tr>
<tr><td><code>ell</code></td><td>体の次数 $\ell$</td><td>必須</td></tr>
<tr><td><code>modulus</code></td><td>$\ell$ 次の既約多項式</td><td>必須</td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td rowspan="2"><code>rs_coefficients</code></td><td><code>raw_bits</code></td><td>Input bit string</td><td>Required</td></tr>
<tr><td><code>ell</code></td><td>Degree of the field $\ell$</td><td>Required</td></tr>
<tr><td rowspan="4"><code>one_bit_extractor</code></td><td><code>coefficients</code></td><td>Return value of <code>rs_coefficients</code></td><td>Required</td></tr>
<tr><td><code>sub_seed</code></td><td>Seed $y_{S_i}$ of length $2\ell$</td><td>Required</td></tr>
<tr><td><code>ell</code></td><td>Degree of the field $\ell$</td><td>Required</td></tr>
<tr><td><code>modulus</code></td><td>Irreducible polynomial of degree $\ell$</td><td>Required</td></tr>
</tbody>
</table>

In [ ]:
def rs_coefficients(raw_bits, ell):
    x = binary_vector(raw_bits)
    ell = positive_integer(ell, "ell")
    blocks = [bits_to_integer(x[j:j + ell]) for j in range(0, len(x), ell)]
    return blocks[::-1]     # the first block has the highest degree -> reverse to list from the lowest degree


def one_bit_extractor(coefficients, sub_seed, ell, modulus):
    y = binary_vector(sub_seed, "sub_seed")
    if len(y) != 2 * ell:
        raise ValueError(f"one-bit seed length must be {2 * ell}")
    alpha = bits_to_integer(y[:ell])   # first ell bits
    beta = bits_to_integer(y[ell:])    # last ell bits
    # Hadamard code: <β, p_α(x)> mod 2
    return (evaluate_polynomial(coefficients, alpha, modulus) & beta).bit_count() & 1

## 5. weak design

<!-- 手法 C の基本の design とブロック weak design を実装します。`block_sizes` は、$2e$ の有理数の上限・下限を `_e_bounds` で作り、段数と段の区切りが両方で同じになることを確かめてから返します（同じにならない場合は項数を増やして計算し直します）。上限には $\sum_{j > N} 1/j! \le \frac{N+2}{(N+1)!\,(N+1)}$ を使います。 -->

We implement the basic design and the block weak design of Method C. `block_sizes` builds rational upper and lower bounds on $2e$ with `_e_bounds`, and returns the result only after checking that both give the same number of stages and the same stage boundaries (if they do not, it recomputes with more terms). The upper bound uses $\sum_{j > N} 1/j! \le \frac{N+2}{(N+1)!\,(N+1)}$.

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 目的 | 戻り値・処理 | 使用例 |
|---|---|---|---|
| `weak_design_set` | 番号 `index` の多項式 $p$ に対応する、基本の weak design の集合 $S_p$（シードの位置）を計算することを目的とした関数です。 | 多項式 `index` の集合のシードの位置（長さ $t$ の `np.int64` の配列） | `weak_design_set(5, 8, 8, 0b1011)` |
| `_e_bounds` | 段の区切りを有理数で厳密に計算するために、Taylor 展開から $2e$ の下限と上限を求めることを目的とした関数です。 | $2e$ の（下限, 上限）の `Fraction` のタプル | `_e_bounds(24)` |
| `block_sizes` | ブロック weak design で、各段に割り当てる集合の数 $(m_0, \ldots, m_L)$ を決めることを目的とした関数です。 | 各段の集合の数 $(m_0, \ldots, m_L)$ のタプル | `block_sizes(10_000, 512)` |
| `block_position` | 出力の `index` ビット目が、どの段の何番目の集合に対応するかを求めることを目的とした関数です。 | 出力の `index` ビット目の（段, 段の中の番号） | `block_position(1700, block_sizes(10_000, 512))` |
| `design_positions` | 出力の `index` ビット目の計算に使うシードの位置 $S_i$ を、段の位置まで含めて計算することを目的とした関数です。 | 出力の `index` ビット目に使うシードの位置 $S_i$（長さ $t$） | `design_positions(1700, sizes, t, q, modulus)` |
-->

| Function | Purpose | Return value / behavior | Example |
|---|---|---|---|
| `weak_design_set` | Computes the set $S_p$ (seed positions) of the basic weak design corresponding to the polynomial $p$ with number `index`. | Seed positions of the set for polynomial `index` (`np.int64` array of length $t$) | `weak_design_set(5, 8, 8, 0b1011)` |
| `_e_bounds` | Obtains lower and upper bounds on $2e$ from the Taylor expansion, to compute the stage boundaries exactly with rational numbers. | Tuple of `Fraction`s (lower bound, upper bound) on $2e$ | `_e_bounds(24)` |
| `block_sizes` | Determines the numbers of sets $(m_0, \ldots, m_L)$ assigned to each stage of the block weak design. | Tuple of the numbers of sets in each stage $(m_0, \ldots, m_L)$ | `block_sizes(10_000, 512)` |
| `block_position` | Finds which set in which stage corresponds to output bit `index`. | (stage, number within the stage) for output bit `index` | `block_position(1700, block_sizes(10_000, 512))` |
| `design_positions` | Computes the seed positions $S_i$ used for output bit `index`, including the offset of the stage. | Seed positions $S_i$ used for output bit `index` (length $t$) | `design_positions(1700, sizes, t, q, modulus)` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td rowspan="4"><code>weak_design_set</code></td><td><code>index</code></td><td>多項式の番号（0 以上の整数）</td><td>必須</td></tr>
<tr><td><code>t</code></td><td>集合の大きさ（$1 \le t \le q$）</td><td>必須</td></tr>
<tr><td><code>q</code></td><td>体の大きさ（2 のべき）</td><td>必須</td></tr>
<tr><td><code>modulus</code></td><td>$\log_2 q$ 次の既約多項式</td><td>必須</td></tr>
<tr><td><code>_e_bounds</code></td><td><code>terms</code></td><td>Taylor 展開の項数</td><td>必須</td></tr>
<tr><td rowspan="2"><code>block_sizes</code></td><td><code>m</code></td><td>出力のビット数</td><td>必須</td></tr>
<tr><td><code>q</code></td><td>体の大きさ（8 以上の 2 のべき）</td><td>必須</td></tr>
<tr><td rowspan="2"><code>block_position</code></td><td><code>index</code></td><td>出力のビット番号（0 以上 $m$ 未満）</td><td>必須</td></tr>
<tr><td><code>sizes</code></td><td><code>block_sizes</code> の戻り値</td><td>必須</td></tr>
<tr><td rowspan="5"><code>design_positions</code></td><td><code>index</code></td><td>出力のビット番号</td><td>必須</td></tr>
<tr><td><code>sizes</code></td><td><code>block_sizes</code> の戻り値</td><td>必須</td></tr>
<tr><td><code>t</code></td><td>集合の大きさ</td><td>必須</td></tr>
<tr><td><code>q</code></td><td>体の大きさ</td><td>必須</td></tr>
<tr><td><code>modulus</code></td><td>$\log_2 q$ 次の既約多項式</td><td>必須</td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td rowspan="4"><code>weak_design_set</code></td><td><code>index</code></td><td>Number of the polynomial (integer of at least 0)</td><td>Required</td></tr>
<tr><td><code>t</code></td><td>Size of the set ($1 \le t \le q$)</td><td>Required</td></tr>
<tr><td><code>q</code></td><td>Size of the field (a power of 2)</td><td>Required</td></tr>
<tr><td><code>modulus</code></td><td>Irreducible polynomial of degree $\log_2 q$</td><td>Required</td></tr>
<tr><td><code>_e_bounds</code></td><td><code>terms</code></td><td>Number of terms of the Taylor expansion</td><td>Required</td></tr>
<tr><td rowspan="2"><code>block_sizes</code></td><td><code>m</code></td><td>Number of output bits</td><td>Required</td></tr>
<tr><td><code>q</code></td><td>Size of the field (a power of 2, at least 8)</td><td>Required</td></tr>
<tr><td rowspan="2"><code>block_position</code></td><td><code>index</code></td><td>Output bit number (at least 0 and less than $m$)</td><td>Required</td></tr>
<tr><td><code>sizes</code></td><td>Return value of <code>block_sizes</code></td><td>Required</td></tr>
<tr><td rowspan="5"><code>design_positions</code></td><td><code>index</code></td><td>Output bit number</td><td>Required</td></tr>
<tr><td><code>sizes</code></td><td>Return value of <code>block_sizes</code></td><td>Required</td></tr>
<tr><td><code>t</code></td><td>Size of the set</td><td>Required</td></tr>
<tr><td><code>q</code></td><td>Size of the field</td><td>Required</td></tr>
<tr><td><code>modulus</code></td><td>Irreducible polynomial of degree $\log_2 q$</td><td>Required</td></tr>
</tbody>
</table>

In [ ]:
def weak_design_set(index, t, q, modulus):
    if index < 0 or not 1 <= t <= q or q & (q - 1):
        raise ValueError("invalid polynomial weak-design parameters")
    if modulus.bit_length() - 1 != q.bit_length() - 1:
        raise ValueError("design modulus degree does not match q")
    # the digits of index written in base q are the coefficients (the ones digit is the constant term)
    coefficients = []
    quotient = index
    while quotient:
        coefficients.append(quotient % q)
        quotient //= q
    if not coefficients:
        coefficients = [0]
    # map the point (u, p(u)) to the seed position u*q + p(u)
    return np.fromiter((u * q + evaluate_polynomial(coefficients, u, modulus) for u in range(t)),
                       dtype=np.int64, count=t)


def _e_bounds(terms):
    factorial = 1
    lower = Fraction(1)
    for j in range(1, terms + 1):
        factorial *= j
        lower += Fraction(1, factorial)
    tail = Fraction(terms + 2, factorial * (terms + 1) * (terms + 1))   # upper bound on the sum of the remaining terms
    return 2 * lower, 2 * (lower + tail)


@lru_cache(maxsize=256)
def block_sizes(m, q):
    m = positive_integer(m, "m")
    q = positive_integer(q, "q")
    if q < 8 or q & (q - 1):
        raise ValueError("q must be a power of two >= 8")
    if m <= q:
        return (m,)   # only constant polynomials, so the sets do not overlap each other
    for terms in (24, 40, 64, 96):
        lower, upper = _e_bounds(terms)          # lower <= 2e <= upper
        a_low, a_high = 1 - 1 / lower, 1 - 1 / upper
        power_low, power_high = a_low, a_high
        levels = 1
        ambiguous = False
        # Eq. (3): the smallest L >= 1 with r + (m - r)(1 - 1/r)^L <= q
        while upper + (m - upper) * power_high > q:
            if lower + (m - lower) * power_low <= q:
                ambiguous = True                  # the upper and lower bounds on 2e disagree
                break
            levels += 1
            power_low *= a_low
            power_high *= a_high
        if ambiguous:
            continue
        sizes, total = [], 0
        power_low, power_high = Fraction(1), Fraction(1)
        for _ in range(levels):
            power_low *= a_low
            power_high *= a_high
            # round up the lower and upper bounds of the cumulative sum (m - r)(1 - (1 - 1/r)^(i+1))
            cumulative_lo = (m - upper) * (1 - power_high)
            cumulative_hi = (m - lower) * (1 - power_low)
            ceil_lo = -(-cumulative_lo.numerator // cumulative_lo.denominator)
            ceil_hi = -(-cumulative_hi.numerator // cumulative_hi.denominator)
            if ceil_lo != ceil_hi:
                ambiguous = True                  # the rounded-up value is not determined
                break
            sizes.append(ceil_lo - total)
            total = ceil_lo
        if ambiguous:
            continue
        sizes.append(m - total)                   # last stage (constant polynomials)
        if min(sizes) < 0 or sizes[-1] > q or sum(sizes) != m:
            raise ArithmeticError("block partition certificate failed")
        return tuple(sizes)
    raise ArithmeticError("Cannot certify block ceilings for 2e")


def block_position(index, sizes):
    cumulative, total = [], 0
    for size in sizes:
        total += size
        cumulative.append(total)
    if not isinstance(index, (int, np.integer)) or not 0 <= index < total:
        raise ValueError("output index outside block partition")
    block = bisect_right(cumulative, index)
    return block, int(index - (cumulative[block - 1] if block else 0))


def design_positions(index, sizes, t, q, modulus):
    block, local_index = block_position(index, sizes)
    return weak_design_set(local_index, t, q, modulus) + block * q * q   # stage i starts at seed position i*q^2

<!-- ## 6. パラメータを決める -->

## 6. Determining the parameters

<!-- 手法 D の表のとおりにパラメータを決めます。浮動小数点数の丸めで条件が甘くならないように、`required_min_entropy` は計算結果を 1 目盛りだけ大きくし、`field_degree` は $\varepsilon$ を有理数に直して整数の計算で切り上げます。 -->

We determine the parameters as in the table of Method D. So that floating-point rounding does not loosen the conditions, `required_min_entropy` increases the result by one unit in the last place, and `field_degree` converts $\varepsilon$ to a rational number and rounds up with integer arithmetic.

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 目的 | 戻り値・処理 | 使用例 |
|---|---|---|---|
| `required_min_entropy` | 出力 $m$ ビットを誤差 $\varepsilon$ で得るために必要な、入力の min-entropy を計算することを目的とした関数です。 | 出力 $m$ ビットに必要な min-entropy $m + 4\log_2(m/\varepsilon) + 6$（float） | `required_min_entropy(9000, 2.0 ** -100)` |
| `field_degree` | 1 ビット抽出器で使う有限体の次数 $\ell$ を、整数演算で厳密に計算することを目的とした関数です。 | $\ell = \lceil \log_2(4nm^2/\varepsilon^2) \rceil$（int） | `field_degree(10_000, 9000, 2.0 ** -100)` |
| `plan_trevisan` | $n$・$h$・$\varepsilon$ から、出力の長さ $m$、体の次数 $\ell$、シードの長さ $d$ など、抽出に必要なパラメータ一式を決めることを目的とした関数です。 | パラメータの辞書（キーは下の表） | `plan_trevisan(10_000, 0.97, 2.0 ** -100)` |
-->

| Function | Purpose | Return value / behavior | Example |
|---|---|---|---|
| `required_min_entropy` | Computes the min-entropy of the input needed to obtain $m$ output bits with error $\varepsilon$. | Min-entropy $m + 4\log_2(m/\varepsilon) + 6$ needed for $m$ output bits (float) | `required_min_entropy(9000, 2.0 ** -100)` |
| `field_degree` | Computes the degree $\ell$ of the finite field used by the one-bit extractor exactly, with integer arithmetic. | $\ell = \lceil \log_2(4nm^2/\varepsilon^2) \rceil$ (int) | `field_degree(10_000, 9000, 2.0 ** -100)` |
| `plan_trevisan` | Determines the full set of parameters needed for extraction, such as the output length $m$, the field degree $\ell$, and the seed length $d$, from $n$, $h$, and $\varepsilon$. | Dictionary of parameters (keys in the table below) | `plan_trevisan(10_000, 0.97, 2.0 ** -100)` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td rowspan="2"><code>required_min_entropy</code></td><td><code>m</code></td><td>出力のビット数</td><td>必須</td></tr>
<tr><td><code>epsilon</code></td><td>全体の誤差 $\varepsilon$</td><td>必須</td></tr>
<tr><td rowspan="3"><code>field_degree</code></td><td><code>n</code></td><td>入力のビット数</td><td>必須</td></tr>
<tr><td><code>m</code></td><td>出力のビット数</td><td>必須</td></tr>
<tr><td><code>epsilon</code></td><td>全体の誤差</td><td>必須</td></tr>
<tr><td rowspan="4"><code>plan_trevisan</code></td><td><code>input_length</code></td><td>入力のビット数 $n$</td><td>必須</td></tr>
<tr><td><code>min_entropy_per_bit</code></td><td>1 ビットあたりの min-entropy $h$</td><td>必須</td></tr>
<tr><td><code>epsilon</code></td><td>全体の誤差 $\varepsilon$（0 < ε < 1）</td><td>必須</td></tr>
<tr><td><code>output_length</code></td><td>出力のビット数。<code>None</code> なら最大値</td><td><code>None</code></td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td rowspan="2"><code>required_min_entropy</code></td><td><code>m</code></td><td>Number of output bits</td><td>Required</td></tr>
<tr><td><code>epsilon</code></td><td>Total error $\varepsilon$</td><td>Required</td></tr>
<tr><td rowspan="3"><code>field_degree</code></td><td><code>n</code></td><td>Number of input bits</td><td>Required</td></tr>
<tr><td><code>m</code></td><td>Number of output bits</td><td>Required</td></tr>
<tr><td><code>epsilon</code></td><td>Total error</td><td>Required</td></tr>
<tr><td rowspan="4"><code>plan_trevisan</code></td><td><code>input_length</code></td><td>Number of input bits $n$</td><td>Required</td></tr>
<tr><td><code>min_entropy_per_bit</code></td><td>Min-entropy per bit $h$</td><td>Required</td></tr>
<tr><td><code>epsilon</code></td><td>Total error $\varepsilon$ (0 < ε < 1)</td><td>Required</td></tr>
<tr><td><code>output_length</code></td><td>Number of output bits. <code>None</code> means the maximum</td><td><code>None</code></td></tr>
</tbody>
</table>

<br>

<!-- `plan_trevisan` の戻り値の主なキーは次のとおりです。 -->

The main keys of the dictionary returned by `plan_trevisan` are as follows.

<!--
| キー | 意味 |
|---|---|
| `input_length_n`, `min_entropy_rate`, `total_min_entropy_k`, `epsilon` | $n$, $h$, $k = nh$, $\varepsilon$ |
| `output_length_m`, `maximum_output_length_m` | 出力のビット数 $m$ と、その最大値 |
| `required_min_entropy`, `one_bit_error` | 出力 $m$ ビットに必要な min-entropy、$\varepsilon_1 = \varepsilon/m$ |
| `field_degree_ell`, `symbols_s`, `padding_bits` | $\ell$、ブロック数 $s$、最後のブロックで 0 を埋めるビット数 |
| `one_bit_seed_length_t`, `design_field_q` | $t = 2\ell$、design の体の大きさ $q$ |
| `block_sizes`, `n_blocks`, `seed_length` | 各段の集合の数、段の数 $L + 1$、シードの長さ $d$ |
-->

| Key | Meaning |
|---|---|
| `input_length_n`, `min_entropy_rate`, `total_min_entropy_k`, `epsilon` | $n$, $h$, $k = nh$, $\varepsilon$ |
| `output_length_m`, `maximum_output_length_m` | Number of output bits $m$ and its maximum |
| `required_min_entropy`, `one_bit_error` | Min-entropy needed for $m$ output bits, and $\varepsilon_1 = \varepsilon/m$ |
| `field_degree_ell`, `symbols_s`, `padding_bits` | $\ell$, the number of blocks $s$, and the number of 0 bits padded in the last block |
| `one_bit_seed_length_t`, `design_field_q` | $t = 2\ell$ and the size $q$ of the design's field |
| `block_sizes`, `n_blocks`, `seed_length` | Number of sets in each stage, the number of stages $L + 1$, and the seed length $d$ |

In [ ]:
def required_min_entropy(m, epsilon):
    return math.nextafter(m + 4 * (math.log2(m) - math.log2(epsilon)) + 6, math.inf)


def field_degree(n, m, epsilon):
    # ell = ceil(log2(4 n m^2 / epsilon^2)): convert epsilon to a rational number and compute with integers
    numerator, denominator = float(epsilon).as_integer_ratio()
    top = 4 * n * m * m * denominator * denominator
    bottom = numerator * numerator
    ell = max(0, top.bit_length() - bottom.bit_length())
    if (bottom << ell) < top:
        ell += 1
    return ell


def plan_trevisan(input_length, min_entropy_per_bit, epsilon, output_length=None):
    n = positive_integer(input_length, "input_length")
    h, epsilon = float(min_entropy_per_bit), float(epsilon)
    if not math.isfinite(h) or not 0 <= h <= 1:
        raise ValueError("min_entropy_per_bit must be finite and in [0,1]")
    if not math.isfinite(epsilon) or not 0 < epsilon < 1:
        raise ValueError("epsilon must be finite and in (0,1)")
    k = n * h
    # required_min_entropy(m) increases with m, so find the largest m satisfying the condition by binary search
    low, high = 0, math.floor(k)
    while low < high:
        middle = (low + high + 1) // 2
        if required_min_entropy(middle, epsilon) <= k:
            low = middle
        else:
            high = middle - 1
    maximum = low
    if maximum == 0:
        raise ValueError(f"No certified positive output: k={k:g}, m=1 needs k>={required_min_entropy(1, epsilon):g}")
    m = maximum if output_length is None else positive_integer(output_length, "output_length")
    if m > maximum:
        raise ValueError(f"output_length must be an integer in [1,{maximum}]")
    ell = field_degree(n, m, epsilon)
    t = 2 * ell
    q = 1 << (t - 1).bit_length()    # the smallest power of 2 that is at least t
    if q < 16:
        raise ValueError("This design certificate requires q>=16")
    sizes = block_sizes(m, q)
    return {
        "input_length_n": n, "min_entropy_rate": h, "total_min_entropy_k": k, "epsilon": epsilon,
        "output_length_m": m, "maximum_output_length_m": maximum,
        "required_min_entropy": required_min_entropy(m, epsilon),
        "one_bit_error": epsilon / m,
        "field_degree_ell": ell, "symbols_s": (n + ell - 1) // ell, "padding_bits": (-n) % ell,
        "one_bit_seed_length_t": t, "design_field_q": q,
        "block_sizes": list(sizes), "n_blocks": len(sizes),
        "seed_length": len(sizes) * q * q,
    }

In [ ]:
plan = plan_trevisan(raw_bits.size, h_min, EPSILON)
for key, value in plan.items():
    print(f"{key:24s}: {value}")

<!-- ## 7. 抽出する -->

## 7. Extracting

<!-- ### 7.1 抽出の関数（Python 版） -->

### 7.1 Extraction functions (Python version)

<!-- `trevisan_extraction` は、6. のパラメータを決め、出力の各ビット $i$ について design のシードの位置 $S_i$ を求めて、1 ビット抽出器 $C(x, y_{S_i})$ を計算します。シードを与えない場合は `generate_seed` で作ります。 -->

`trevisan_extraction` determines the parameters of 6., finds the design's seed positions $S_i$ for each output bit $i$, and computes the one-bit extractor $C(x, y_{S_i})$. If no seed is given, one is generated with `generate_seed`.

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 目的 | 戻り値・処理 | 使用例 |
|---|---|---|---|
| `generate_seed` | 抽出に使う指定の長さのシードを生成することを目的とした関数です。 | 長さ `length` のシード（`np.uint8` の配列） | `generate_seed(100)` |
| `trevisan_extraction` | パラメータの決定から全出力ビットの計算までをまとめて行い、Trevisan's extractor の抽出を Python で実行することを目的とした関数です。 | `(output_bits, seed, info)` のタプル。`info` は `plan_trevisan` の辞書に既約多項式と計算時間を加えたもの | `trevisan_extraction(raw_bits, h_min, EPSILON)` |
-->

| Function | Purpose | Return value / behavior | Example |
|---|---|---|---|
| `generate_seed` | Generates a seed of the specified length for extraction. | Seed of length `length` (`np.uint8` array) | `generate_seed(100)` |
| `trevisan_extraction` | Performs everything from determining the parameters to computing all output bits, running Trevisan's extractor in Python. | Tuple `(output_bits, seed, info)`. `info` is the dictionary from `plan_trevisan` with the irreducible polynomials and the computation time added | `trevisan_extraction(raw_bits, h_min, EPSILON)` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td rowspan="2"><code>generate_seed</code></td><td><code>length</code></td><td>シードのビット数</td><td>必須</td></tr>
<tr><td><code>rng</code></td><td><code>None</code>（OS の暗号用乱数）、または乱数シードの整数・<code>np.random.Generator</code>（確認用）</td><td><code>None</code></td></tr>
<tr><td rowspan="6"><code>trevisan_extraction</code></td><td><code>raw_bits</code></td><td>入力のビット列</td><td>必須</td></tr>
<tr><td><code>min_entropy_per_bit</code></td><td>1 ビットあたりの min-entropy $h$</td><td>必須</td></tr>
<tr><td><code>epsilon</code></td><td>全体の誤差 $\varepsilon$</td><td>必須</td></tr>
<tr><td><code>seed</code></td><td>長さ $d$ のシード。<code>None</code> なら新しく作る</td><td><code>None</code></td></tr>
<tr><td><code>output_length</code></td><td>出力のビット数。<code>None</code> なら最大値</td><td><code>None</code></td></tr>
<tr><td><code>rng</code></td><td><code>seed=None</code> のときのシードの作り方（<code>generate_seed</code> と同じ）</td><td><code>None</code></td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td rowspan="2"><code>generate_seed</code></td><td><code>length</code></td><td>Number of seed bits</td><td>Required</td></tr>
<tr><td><code>rng</code></td><td><code>None</code> (the OS's cryptographic random numbers), or an integer random seed or <code>np.random.Generator</code> (for checking)</td><td><code>None</code></td></tr>
<tr><td rowspan="6"><code>trevisan_extraction</code></td><td><code>raw_bits</code></td><td>Input bit string</td><td>Required</td></tr>
<tr><td><code>min_entropy_per_bit</code></td><td>Min-entropy per bit $h$</td><td>Required</td></tr>
<tr><td><code>epsilon</code></td><td>Total error $\varepsilon$</td><td>Required</td></tr>
<tr><td><code>seed</code></td><td>Seed of length $d$. <code>None</code> means generate a new one</td><td><code>None</code></td></tr>
<tr><td><code>output_length</code></td><td>Number of output bits. <code>None</code> means the maximum</td><td><code>None</code></td></tr>
<tr><td><code>rng</code></td><td>How the seed is generated when <code>seed=None</code> (same as <code>generate_seed</code>)</td><td><code>None</code></td></tr>
</tbody>
</table>

In [ ]:
def generate_seed(length, rng=None):
    if rng is None:
        random_bytes = secrets.token_bytes((length + 7) // 8)
        return np.unpackbits(np.frombuffer(random_bytes, dtype=np.uint8))[:length].copy()
    return np.random.default_rng(rng).integers(0, 2, size=length, dtype=np.uint8)


def _prepare_extraction(raw_bits, min_entropy_per_bit, epsilon, seed, output_length, rng):
    # preparation shared by the Python and fast versions: parameters, irreducible polynomials, seed
    x = binary_vector(raw_bits)
    info = plan_trevisan(len(x), min_entropy_per_bit, epsilon, output_length=output_length)
    modulus = irreducible_polynomial(info["field_degree_ell"])                          # for GF(2^ell)
    design_modulus = irreducible_polynomial(info["design_field_q"].bit_length() - 1)   # for GF(q) (design)
    info["irreducible_polynomial"] = hex(modulus)
    info["design_irreducible_polynomial"] = hex(design_modulus)
    if seed is None:
        y = generate_seed(info["seed_length"], rng)
        info["seed_source"] = "secrets.token_bytes" if rng is None else "numpy (reproducible, for checking only)"
    else:
        y = binary_vector(seed, "seed").copy()
        if len(y) != info["seed_length"]:
            raise ValueError(f"seed length must be {info['seed_length']}, got {len(y)}")
        info["seed_source"] = "provided"
    return x, y, info, modulus, design_modulus


def trevisan_extraction(raw_bits, min_entropy_per_bit, epsilon, seed=None, output_length=None, rng=None):
    x, y, info, modulus, design_modulus = _prepare_extraction(raw_bits, min_entropy_per_bit, epsilon,
                                                              seed, output_length, rng)
    ell, t, q = info["field_degree_ell"], info["one_bit_seed_length_t"], info["design_field_q"]
    sizes = info["block_sizes"]
    coefficients = rs_coefficients(x, ell)
    started = time.perf_counter()
    output = np.empty(info["output_length_m"], dtype=np.uint8)
    for i in range(len(output)):
        positions = design_positions(i, sizes, t, q, design_modulus)              # S_i
        output[i] = one_bit_extractor(coefficients, y[positions], ell, modulus)    # C(x, y_{S_i})
    info["extraction_s"] = time.perf_counter() - started
    return output, y, info

<!-- ### 7.2 抽出の関数（`numba` を使う高速版） -->

### 7.2 Extraction functions (fast version with `numba`)

<!-- 7.1 と同じ計算を `numba` で高速にします。$\mathrm{GF}(2^\ell)$ の要素を 64 ビットの整数（limb）の配列で表し、Horner 法の掛け算 $v \mapsto v\alpha$ を、$\alpha X^{8j} \cdot (\text{1 バイトの値})$ の表を引いて XOR することで計算します。浮動小数点数は使わないので、出力は 7.1 と同じです。`numba` がない場合、このセルは関数を定義するだけで、7.3 では 7.1 を使います。 -->

We speed up the same computation as in 7.1 with `numba`. An element of $\mathrm{GF}(2^\ell)$ is represented as an array of 64-bit integers (limbs), and the multiplication $v \mapsto v\alpha$ in Horner's method is computed by looking up a table of $\alpha X^{8j} \cdot (\text{a 1-byte value})$ and XORing. No floating-point numbers are used, so the output is the same as in 7.1. If `numba` is not available, this cell only defines the functions, and 7.1 is used in 7.3.

<!-- `modulus_low` は既約多項式 $f$ から最高次の項 $X^\ell$ を除いたもので、$X^\ell \equiv f - X^\ell$ を使って次数を下げるのに使います。 -->

`modulus_low` is the irreducible polynomial $f$ with its highest-degree term $X^\ell$ removed; it is used to reduce the degree via $X^\ell \equiv f - X^\ell$.

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 目的 | 戻り値・処理 | 使用例 |
|---|---|---|---|
| `to_limbs` | $\ell$ ビットの大きな整数を `numba` で扱えるよう、64 ビットずつの配列に分割することを目的とした関数です。 | 64 ビットずつに分けた `np.uint64` の配列（長さ `count`） | `to_limbs(2 ** 70 + 5, 2)` |
| `trevisan_extraction_numba` | `trevisan_extraction` と同じ抽出を、`numba` を使って高速に実行することを目的とした関数です。 | `(output_bits, seed, info)` のタプル（`trevisan_extraction` と同じ） | `trevisan_extraction_numba(raw_bits, h_min, EPSILON)` |
-->

| Function | Purpose | Return value / behavior | Example |
|---|---|---|---|
| `to_limbs` | Splits a large $\ell$-bit integer into an array of 64-bit pieces so that `numba` can handle it. | `np.uint64` array of 64-bit pieces (length `count`) | `to_limbs(2 ** 70 + 5, 2)` |
| `trevisan_extraction_numba` | Performs the same extraction as `trevisan_extraction`, quickly, using `numba`. | Tuple `(output_bits, seed, info)` (same as `trevisan_extraction`) | `trevisan_extraction_numba(raw_bits, h_min, EPSILON)` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td rowspan="2"><code>to_limbs</code></td><td><code>value</code></td><td>0 以上の整数</td><td>必須</td></tr>
<tr><td><code>count</code></td><td>limb の個数</td><td>必須</td></tr>
<tr><td rowspan="6"><code>trevisan_extraction_numba</code></td><td><code>raw_bits</code></td><td>入力のビット列</td><td>必須</td></tr>
<tr><td><code>min_entropy_per_bit</code></td><td>$h$</td><td>必須</td></tr>
<tr><td><code>epsilon</code></td><td>$\varepsilon$</td><td>必須</td></tr>
<tr><td><code>seed</code></td><td>長さ $d$ のシード。<code>None</code> なら新しく作る</td><td><code>None</code></td></tr>
<tr><td><code>output_length</code></td><td>出力のビット数</td><td><code>None</code></td></tr>
<tr><td><code>rng</code></td><td>シードの作り方</td><td><code>None</code></td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td rowspan="2"><code>to_limbs</code></td><td><code>value</code></td><td>Integer of at least 0</td><td>Required</td></tr>
<tr><td><code>count</code></td><td>Number of limbs</td><td>Required</td></tr>
<tr><td rowspan="6"><code>trevisan_extraction_numba</code></td><td><code>raw_bits</code></td><td>Input bit string</td><td>Required</td></tr>
<tr><td><code>min_entropy_per_bit</code></td><td>$h$</td><td>Required</td></tr>
<tr><td><code>epsilon</code></td><td>$\varepsilon$</td><td>Required</td></tr>
<tr><td><code>seed</code></td><td>Seed of length $d$. <code>None</code> means generate a new one</td><td><code>None</code></td></tr>
<tr><td><code>output_length</code></td><td>Number of output bits</td><td><code>None</code></td></tr>
<tr><td><code>rng</code></td><td>How the seed is generated</td><td><code>None</code></td></tr>
</tbody>
</table>

<br>

<!-- `design_set_numba(index, t, q, modulus)` は `weak_design_set` と、`rs_hadamard_numba(coefficients, alpha, beta, ell, modulus_low)` は `one_bit_extractor` と同じ計算です（係数・$\alpha$・$\beta$・`modulus_low` は `to_limbs` で limb の配列にしたもの）。 -->

`design_set_numba(index, t, q, modulus)` performs the same computation as `weak_design_set`, and `rs_hadamard_numba(coefficients, alpha, beta, ell, modulus_low)` the same as `one_bit_extractor` (the coefficients, $\alpha$, $\beta$, and `modulus_low` are converted to limb arrays with `to_limbs`).

In [ ]:
def to_limbs(value, count):
    return np.array([(value >> (64 * j)) & ((1 << 64) - 1) for j in range(count)], dtype=np.uint64)


if NUMBA_AVAILABLE:
    @njit
    def design_set_numba(index, t, q, modulus):
        coefficients = []
        while index:
            coefficients.append(index % q)
            index //= q
        result = np.empty(t, dtype=np.int64)
        for u in range(t):
            value = 0
            for j in range(len(coefficients) - 1, -1, -1):
                a, b, product = value, u, 0
                while b:                      # multiplication in GF(q): value * u
                    if b & 1:
                        product ^= a
                    b >>= 1
                    a <<= 1
                    if a & q:
                        a ^= modulus
                value = product ^ coefficients[j]
            result[u] = u * q + value
        return result

    @njit
    def rs_hadamard_numba(coefficients, alpha, beta, ell, modulus_low):
        limbs = len(alpha)
        nbytes = (ell + 7) // 8
        table = np.zeros((nbytes, 256, limbs), dtype=np.uint64)   # table[j, byte] = alpha * X^(8j) * byte
        monomial = alpha.copy()
        one = np.uint64(1)
        for bit in range(nbytes * 8):
            for word in range(limbs):
                table[bit // 8, 1 << (bit % 8), word] = monomial[word]
            high = (monomial[(ell - 1) // 64] >> np.uint64((ell - 1) % 64)) & one
            carry = np.uint64(0)
            for word in range(limbs):             # monomial *= X
                next_carry = monomial[word] >> np.uint64(63)
                monomial[word] = (monomial[word] << one) | carry
                carry = next_carry
            if ell % 64:
                monomial[-1] &= (one << np.uint64(ell % 64)) - one
            if high:                              # replace X^ell with f - X^ell
                for word in range(limbs):
                    monomial[word] ^= modulus_low[word]
        for byte_index in range(nbytes):          # build the tables for all bytes from the single-bit values
            for value in range(1, 256):
                low = value & -value
                if value == low:
                    continue
                for word in range(limbs):
                    table[byte_index, value, word] = table[byte_index, low, word] ^ table[byte_index, value ^ low, word]
        value = np.zeros(limbs, dtype=np.uint64)
        scratch = np.zeros(limbs, dtype=np.uint64)
        for coefficient in range(len(coefficients) - 1, -1, -1):   # Horner's method
            for word in range(limbs):
                scratch[word] = coefficients[coefficient, word]
            for byte_index in range(nbytes):
                byte = (value[byte_index // 8] >> np.uint64(8 * (byte_index % 8))) & np.uint64(255)
                for word in range(limbs):
                    scratch[word] ^= table[byte_index, byte, word]
            value, scratch = scratch, value
        parity_word = np.uint64(0)
        for word in range(limbs):
            parity_word ^= value[word] & beta[word]
        for shift in (32, 16, 8, 4, 2, 1):
            parity_word ^= parity_word >> np.uint64(shift)
        return int(parity_word & one)


def trevisan_extraction_numba(raw_bits, min_entropy_per_bit, epsilon, seed=None, output_length=None, rng=None):
    if not NUMBA_AVAILABLE:
        raise ImportError("numba is not installed")
    x, y, info, modulus, design_modulus = _prepare_extraction(raw_bits, min_entropy_per_bit, epsilon,
                                                              seed, output_length, rng)
    ell, t, q = info["field_degree_ell"], info["one_bit_seed_length_t"], info["design_field_q"]
    sizes = info["block_sizes"]
    limb_count = (ell + 63) // 64
    coefficient_limbs = np.array([to_limbs(c, limb_count) for c in rs_coefficients(x, ell)], dtype=np.uint64)
    modulus_low = to_limbs(modulus ^ (1 << ell), limb_count)
    started = time.perf_counter()
    output = np.empty(info["output_length_m"], dtype=np.uint8)
    for i in range(len(output)):
        block, local_index = block_position(i, sizes)
        positions = design_set_numba(local_index, t, q, design_modulus) + block * q * q   # S_i
        sub_seed = y[positions]
        alpha = to_limbs(bits_to_integer(sub_seed[:ell]), limb_count)
        beta = to_limbs(bits_to_integer(sub_seed[ell:]), limb_count)
        output[i] = rs_hadamard_numba(coefficient_limbs, alpha, beta, ell, modulus_low)
    info["extraction_s"] = time.perf_counter() - started
    return output, y, info

<!-- ### 7.3 抽出を実行する -->

### 7.3 Running the extraction

<!-- 2. の入力を抽出します。`USE_NUMBA = True` で `numba` があれば 7.2 の高速版（最初の実行ではコンパイルに数秒かかります）、それ以外は 7.1 の Python 版を使います。 -->

We extract from the input of 2. If `USE_NUMBA = True` and `numba` is available, the fast version of 7.2 is used (compilation takes a few seconds on the first run); otherwise the Python version of 7.1 is used.

In [ ]:
USE_NUMBA = True

extract = trevisan_extraction_numba if (USE_NUMBA and NUMBA_AVAILABLE) else trevisan_extraction
extracted, trevisan_seed, extraction_info = extract(raw_bits, h_min, EPSILON, rng=SEED_RNG)
print(f"Function used  : {extract.__name__}")
print(f"Output m       : {extraction_info['output_length_m']} bits (input n = {raw_bits.size})")
print(f"Seed d         : {extraction_info['seed_length']} bits ({extraction_info['seed_source']})")
print(f"Time           : {extraction_info['extraction_s']:.1f} s")
print(f"P(1) of output : {extracted.mean():.5f}")
print("First 40 bits of the output:", "".join(map(str, extracted[:40])))

<!-- ## 8. 抽出結果を保存する -->

## 8. Saving the extraction results

<!-- 1. で `SAVE_RESULTS = True` にした場合は、`RUN_DIR` の次のファイルに保存します（既存のファイルは上書きしません）。 -->

If `SAVE_RESULTS = True` in 1., the results are saved to the following files in `RUN_DIR` (existing files are not overwritten).

<!--
- `input_bits.txt`：抽出の入力のビット列（0 と 1 の文字、改行なし）
- `trevisan_extracted.txt`：抽出したビット列（0 と 1 の文字、改行なし。`load_bits` で読み込めます）
- `trevisan_seed.txt`：シード（同じ形式）
- `trevisan_info.json`：パラメータ（`extraction_info`）
-->

- `input_bits.txt`: the input bit string of the extraction (the characters 0 and 1, no newlines)
- `trevisan_extracted.txt`: the extracted bit string (the characters 0 and 1, no newlines; can be read with `load_bits`)
- `trevisan_seed.txt`: the seed (same format)
- `trevisan_info.json`: the parameters (`extraction_info`)

<!-- 同じ入力・$h$・$\varepsilon$・出力の長さとこのシードを `trevisan_extraction` に渡すと、同じ出力が得られます。 -->

Passing the same input, $h$, $\varepsilon$, output length, and this seed to `trevisan_extraction` gives the same output.

In [ ]:
if RUN_DIR is None:
    print("SAVE_RESULTS = False, so nothing is saved.")
else:
    # Existing files are never overwritten
    save_bits(RUN_DIR / "input_bits.txt", raw_bits)
    save_bits(RUN_DIR / "trevisan_extracted.txt", extracted)
    save_bits(RUN_DIR / "trevisan_seed.txt", trevisan_seed)
    info = dict(extraction_info, input_path=None if INPUT_PATH is None else str(INPUT_PATH),
                simulation=None if INPUT_PATH is not None else dict(
                    n_bits=N_BITS, p1_given_0=P1_GIVEN_0, p0_given_1=P0_GIVEN_1, seed=SIM_SEED),
                h_min_source="MCV" if H_MIN_OVERRIDE is None else "H_MIN_OVERRIDE")
    with (RUN_DIR / "trevisan_info.json").open("x", encoding="utf-8") as f:
        f.write(json.dumps(info, ensure_ascii=False, indent=2) + "\n")
    print("Saved to:", RUN_DIR)